In [ ]:
import torch
import subprocess
import random
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO
import pyautogui
import time

device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

: 

In [ ]:
# Setting model hyperparameters
gamma = 0.99            # Discount factor for future rewards
epsilon = 1.0           # Initial exploration rate
epsilon_decay = 0.995   # Decay rate for the exploration probability per episode
min_epsilon = 0.01      # Minimum exploration rate
learning_rate = 0.001   # Rate at which model changes parameters
num_episodes = 1000     # Number of episodes to engage in

# Action mapping for emulator
ACTION_MAP = {
    'A': 'x',
    'B': 'z',
    'X': 's',
    'Y': 'a',
    'Up': 'up',
    'Down': 'down',
    'Left': 'left',
    'Right': 'right'
}

#### Examining how epsilon (exploration rate) changes through episodes (training)

In [ ]:
# List to store epsilon values
epsilon_values = []

# Simulate epsilon decay over episodes
for episode in range(num_episodes):
    epsilon = max(min_epsilon, epsilon * epsilon_decay)
    epsilon_values.append(epsilon)

# Plotting the epsilon decay
plt.figure(figsize=(10, 6))
plt.plot(epsilon_values, label='Epsilon')
plt.xlabel('Episode')
plt.ylabel('Epsilon Value')
plt.title('Epsilon Decay Over Episodes')
plt.legend()
plt.grid(True)
plt.show()


### Loading the pretrained model

In [ ]:
# Load the trained model
model_path = 'models/pokemon_model_lstm.pth'
model = torch.load(model_path)
model.load_state_dict()
model.eval()

# Load in annotation model
annotation_model_pth = "runs/detect/finalRun/weights/best.pt"
annotation_modelmodel = YOLO(annotation_model_pth)

### Connecting script to emulator (DeSmuME)

In [20]:
# Path to the DeSmuME executable, the ROM file, and desired state
# Won't be starting from scratch to avoid uneccessary cutscenes
desmume_executable = '/Applications/DeSmuME.app/Contents/MacOS/DeSmuME'
pokemon_rom = 'Platinum_Randomized.nds'
state_file = '/Users/scottpitcher/Library/Application Support/DeSmuME/0.9.13/States/Platinum_Randomized.ds4'

# Start DeSmuME emulator
def open_emulator():
    subprocess.Popen([desmume_executable, pokemon_rom])
    # Wait for the emulator to start (this is the amount of time until actions can be sent to emulator after loading screen)
    time.sleep(20) 
    # Load in the state
    key = f'4'
    pyautogui.press('x')
    pyautogui.press(key)
    print(f"Loaded state {key}")

open_emulator()

2024-06-25 12:38:44.446 DeSmuME[3709:7179896] WARNING: Secure coding is not enabled for restorable state! Enable secure coding by implementing NSApplicationDelegate.applicationSupportsSecureRestorableState: and returning YES.


Loaded state 4


Microphone successfully inited.
DeSmuME 0.9.13 ARM64 NEON-A64

ROM game code: CPUE
ROM crc: A95C7305
ROM serial: NTR-CPUE-USA
ROM chipID: 00007FC2
ROM internal name: POKEMON PL
ROM developer: Nintendo

Load cheats: /Users/scottpitcher/Library/Application Support/DeSmuME/0.9.13/Cheats/Platinum_Randomized.dct
Added 2 cheat codes
Slot1 auto-selected device type: Retail MC+ROM
Slot2 auto-selected device type: None (0xFF)
BackupDevice: size = 4 Mbit
CPU mode: Interpreter
Already decrypted.
WIFI: MAC Address = 00:09:BF:9B:27:8F
WIFI: Emulation level is OFF.
Load cheats: /Users/scottpitcher/Library/Application Support/DeSmuME/0.9.13/Cheats/Platinum_Randomized.dct
Added 2 cheat codes
SoftRasterizer: Running using 8 additional threads. (Multithreading enabled.)
Slot1 auto-selected device type: Retail MC+ROM
Slot2 auto-selected device type: None (0xFF)
BackupDevice: size = 4 Mbit
CPU mode: Interpreter
Already decrypted.
WIFI: MAC Address = 00:09:BF:9B:27:8F
WIFI: Emulation level is OFF.
Slot 2: 

In [ ]:
# Functions to interact with the emulator

def get_game_state(output_dir='game_state_screenshots', frame_num=0):
    """Function to capture the current game state from the emulator"""
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    screenshot = pyautogui.screenshot()
    frame_path = os.path.join(output_dir, f'frame_{frame_num}.png')
    screenshot.save(frame_path)
    print(f"Saved game state to {frame_path}")

def perform_action(action):
    """Function to perform the given action in the emulator."""
    if action in ACTION_MAP:
        key = ACTION_MAP[action]
        pyautogui.press(key)
        time.sleep(0.1)  # Adding a short delay to ensure the action is registered

### Defining Rewards

In [ ]:
def calculate_reward(state, action, next_state):
    reward = 0
    if next_state['win']:
        reward += 10
    elif next_state['loss']:
        reward -= 10
    return reward


### Collecting Human Feedback

In [ ]:
def get_human_feedback(states, actions):
    feedback = []
    for state, action in zip(states, actions):
        # Display state and action to the human for feedback
        print(f"State: {state}, Action: {action}")
        fb = input("Is this action correct? (yes/no): ")
        feedback.append(1 if fb.lower() == 'yes' else 0)
    return feedback
